# 🧪 Sesión 18 · Rate Limiting, Throttling y Prevención de Abuso

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo evitas que un script sature tu API?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/recurso   -> 200 OK   (1.ª petición)
GET /api/recurso   -> 200 OK   (2.ª petición)
GET /api/recurso   -> 200 OK   (3.ª petición)
GET /api/recurso   -> 429 Too Many Requests
Retry-After: 60
{"error":"límite de peticiones alcanzado"}
```

_El límite protege el servicio para todos; no es un castigo al cliente._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar un límite de peticiones por cliente.
- Responder 429 con Retry-After.
- Explicar la diferencia entre limitar y bloquear.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Primera petición permitida → HTTP 200
- [ ] Segunda petición permitida → HTTP 200
- [ ] Tercera petición permitida → HTTP 200
- [ ] La cuarta petición se limita → HTTP 429

## 🧩 El reto

Implementa un límite de tres peticiones por minuto por cliente y responde 429 cuando se exceda.

**Requisitos del contrato**

- GET /api/recurso permite tres peticiones por cliente en una ventana de 60 segundos.
- La cuarta petición responde 429.
- La respuesta 429 incluye Retry-After.
- Las tres primeras responden 200 con un JSON simple.
- El contador se reinicia al iniciar la aplicación.

> **Pista conceptual:** El cliente se identifica por IP o por token. Un contador global castiga a todos; un contador por cliente protege a todos.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion18.md` con el cálculo de la ventana.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py
import time
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse

app = FastAPI(title="Tu límite de peticiones se verá así")

# Diccionario en memoria para rastrear las peticiones por IP del cliente
# Estructura: { ip: [timestamp1, timestamp2, ...] }
request_history = {}
WINDOW_SECONDS = 60
MAX_REQUESTS = 3

@app.get("/api/recurso")
async def obtener_recurso(request: Request):
    client_ip = request.client.host if request.client else "127.0.0.1"
    now = time.time()

    # Inicializar el historial de la IP si no existe
    if client_ip not in request_history:
        request_history[client_ip] = []

    # Filtrar peticiones para mantener solo las que están dentro de la ventana de 60 segundos
    request_history[client_ip] = [t for t in request_history[client_ip] if now - t < WINDOW_SECONDS]

    # Verificar si excede el límite
    if len(request_history[client_ip]) >= MAX_REQUESTS:
        # Calcular el tiempo restante para que expire la petición más antigua del bloque
        oldest_request = request_history[client_ip][0]
        retry_after = int(WINDOW_SECONDS - (now - oldest_request))
        if retry_after <= 0:
            retry_after = 1

        return JSONResponse(
            status_code=429,
            content={"error": "límite de peticiones alcanzado"},
            headers={"Retry-After": str(retry_after)}
        )

    # Registrar la petición actual y responder exitosamente
    request_history[client_ip].append(now)
    return {"mensaje": "petición permitida"}


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [4]:
from verificar import probar_http

casos = [{'descripcion': 'Primera petición permitida', 'metodo': 'GET', 'ruta': '/api/recurso', 'estado': 200},
 {'descripcion': 'Segunda petición permitida', 'metodo': 'GET', 'ruta': '/api/recurso', 'estado': 200},
 {'descripcion': 'Tercera petición permitida', 'metodo': 'GET', 'ruta': '/api/recurso', 'estado': 200},
 {'descripcion': 'La cuarta petición se limita',
  'metodo': 'GET',
  'ruta': '/api/recurso',
  'estado': 429,
  'headers_contiene': {'retry-after': ''},
  'json_contiene': {'error': 'límite de peticiones alcanzado'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})

VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. Primera petición permitida
✅ 02. Segunda petición permitida
✅ 03. Tercera petición permitida
✅ 04. La cuarta petición se limita
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion18.md` a `auditorias/auditoria_sesion18.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
